In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, average_precision_score
)
import warnings
import shap 
warnings.filterwarnings('ignore')

f:\app space\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# Telco Customer Churn – Feature Engineering Documentation

## Financial Features

Financial behavior is one of the strongest indicators of customer retention. Customers who spend more money, maintain long-term subscriptions, or exhibit recent billing changes often demonstrate different churn patterns.

This section creates financial features that capture customer value, spending consistency, and pricing dynamics.

### Feature: Average Charge per Month (`AvgChargePerMonth`)

**Business Motivation**

Total charges naturally increase with customer tenure, making direct comparisons between customers difficult. Dividing total charges by tenure provides an estimate of the customer's average monthly spending.

**Hypothesis**

Customers with higher average monthly spending may represent premium users and therefore exhibit different churn behavior.

**Formula**

AvgChargePerMonth = TotalCharges / Tenure

### Feature: Customer Lifetime Value (`CLV`)

**Business Motivation**

Customers who have generated higher cumulative revenue over their relationship with the company generally represent more valuable business assets.

Although this is a simplified approximation, it captures the overall economic value of a customer.

**Hypothesis**

Customers with higher estimated lifetime value are expected to have lower churn probability.

**Formula**

CLV = MonthlyCharges × Tenure

### Feature: Charge Increase Ratio (`Charge_Increase_Ratio`)

**Business Motivation**

A customer's current monthly charge may differ from their historical average due to upgrades, promotions ending, or price adjustments.

This feature measures whether the customer is currently paying more than their historical average.

**Hypothesis**

Large increases in monthly charges may increase customer dissatisfaction and therefore increase churn risk.

**Formula**

Charge_Increase_Ratio = MonthlyCharges / AvgChargePerMonth

### Feature: Recent Price Hike (`Recent_Price_Hike`)

**Business Motivation**

Customers experiencing noticeable price increases may become more sensitive to competitors.

**Hypothesis**

Recent price increases increase the likelihood of customer churn.

---

## Contract and Payment Features

Customer payment behavior reflects commitment, convenience, and contractual obligations.

These features describe payment preferences and identify combinations historically associated with higher churn.

### Feature: Month-to-Month Contract (`IsMonthContract`)

**Business Motivation**

Customers without long-term contracts can leave the company at any time.

**Hypothesis**

Month-to-month customers have significantly higher churn risk.

### Feature: Automatic Payment (`AutoPay`)

**Business Motivation**

Customers enrolled in automatic payments experience fewer billing interruptions and demonstrate stronger commitment.

**Hypothesis**

Automatic payment users are less likely to churn.

### Feature: Paperless Billing and AutoPay Interaction (`Paperless_AutoPay_Interaction`)

**Business Motivation**

Customers who simultaneously adopt paperless billing and automatic payment tend to be digitally engaged.

**Hypothesis**

Digitally engaged customers are expected to be more loyal.

### Feature: High-Risk Payment & Contract Combination (`High_Risk_Payment_Contract`)

**Business Motivation**

Previous business observations indicate that customers using Electronic Check together with Month-to-Month contracts often exhibit elevated churn rates.

**Hypothesis**

This combination represents a high-risk customer segment.

---

## Service Usage Features

Customer engagement is strongly related to the breadth of subscribed services.

The more services a customer actively uses, the more difficult switching providers becomes.

### Feature: Number of Add-on Services (`NumAddOnServices`)

**Business Motivation**

Customers using multiple additional services are generally more integrated into the company's ecosystem.

**Hypothesis**

The probability of churn decreases as service adoption increases.

### Feature: Multiple Phone Lines (`HasMultipleLines`)

**Business Motivation**

Customers with multiple phone lines usually have more complex service needs.

**Hypothesis**

Higher service dependency reduces churn.

### Feature: Charges Per Service (`ChargesPerService`)

**Business Motivation**

Normalizing monthly charges by the number of subscribed services estimates the average cost paid per service.

**Hypothesis**

Customers paying disproportionately high prices per service may be more likely to churn.

---

## Family and Demographic Features

Household characteristics often influence customer retention.

Family-related subscriptions and demographic factors provide valuable behavioral information.

### Feature: Family Presence (`HasFamily`)

**Business Motivation**

Customers living with partners or dependents often maintain more stable service subscriptions.

**Hypothesis**

Family presence reduces churn probability.

### Feature: Family Risk (`FamilyRisk`)

**Business Motivation**

Senior citizens without family support may require different customer service experiences.

**Hypothesis**

Senior customers without family support exhibit higher churn risk.

---

## Tenure Features

Customer tenure is one of the strongest predictors of churn.

Longer relationships generally indicate greater loyalty and satisfaction.

### Feature: Tenure Group (`TenureGroup`)

**Business Motivation**

Customers at different lifecycle stages often display distinct churn behaviors.

**Hypothesis**

Early-stage customers are more likely to leave than long-term customers.

### Feature: New Customer (`IsNewCustomer`)

**Business Motivation**

The first months of service represent the highest-risk period for churn.

**Hypothesis**

New customers are more likely to churn.

### Feature: Loyal Customer (`IsLoyalCustomer`)

**Business Motivation**

Customers who remain subscribed for long periods usually demonstrate stronger loyalty.

**Hypothesis**

Loyal customers have significantly lower churn probability.

---

## Protection and Entertainment Features

Additional security and entertainment services increase customer engagement and create switching costs.

* **HasProtection** → Adoption of security-related services.
* **FullProtectionBundle** → Customer subscribes to all protection services.
* **UsesStreaming** → Customer actively uses streaming services.
* **FiberUser** → Customer uses Fiber Optic internet, which may be associated with different pricing and service quality.

---

## Service Type Categorization

Customers may subscribe to phone services, internet services, both, or neither.

Categorizing service portfolios helps identify behavioral differences across customer segments.

---

## Composite Engagement Score

### Feature: Engagement Score (`EngagementScore`)

**Business Motivation**

Customer engagement is multidimensional and cannot be represented by a single variable.

This feature combines service adoption, protection services, streaming usage, and payment commitment into a unified engagement metric.

**Hypothesis**

Customers with higher engagement scores are expected to exhibit substantially lower churn rates.

**Formula**

EngagementScore =
NumAddOnServices
+ HasProtection
+ UsesStreaming
+ AutoPay

In [3]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


class TelcoFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Feature engineering transformer for Telco churn dataset.
    
    Creates:
    - Engagement and loyalty features (tenure groups, new/loyal customer flags)
    - Service usage features (add-ons, streaming, protection bundles)
    - Payment and contract risk features (auto-pay, high-risk combinations)
    - Derived financial features (AvgChargePerMonth, CLV, price hike flag)
    - Family and demographic risk features
    - Service type categorization (Both/PhoneOnly/InternetOnly/Neither)
    - Engagement score summarizing multiple behaviors
    """
    
    def __init__(
        self,
        new_customer_threshold=6,
        loyal_customer_threshold=24,
        price_hike_ratio=1.05,
        tenure_bins=None,
        tenure_labels=None,
    ):
        self.new_customer_threshold = new_customer_threshold
        self.loyal_customer_threshold = loyal_customer_threshold
        self.price_hike_ratio = price_hike_ratio
        
        # Default tenure grouping
        self.tenure_bins = tenure_bins or [0, 12, 24, 48, np.inf]
        self.tenure_labels = tenure_labels or ["0-12", "12-24", "24-48", "48+"]

        # Column groups for easier maintenance
        self._addon_cols = [
            "OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        self._protection_cols = ["OnlineSecurity", "DeviceProtection", "TechSupport"]
        self._streaming_cols = ["StreamingTV", "StreamingMovies"]

    def fit(self, X, y=None):
        # No fitting needed; just return self
        return self

    def transform(self, X):
        X = X.copy()
        
        # 1. Financial features
        X = self._add_financial_features(X)
        
        # 2. Contract & payment features
        X = self._add_contract_payment_features(X)
        
        # 3. Service usage & engagement features
        X = self._add_service_usage_features(X)
        
        # 4. Family & demographic risk features
        X = self._add_family_demographic_features(X)
        
        # 5. Tenure-based features
        X = self._add_tenure_features(X)
        
        # 6. Protection & streaming bundles
        X = self._add_protection_streaming_features(X)
        
        # 7. Service type categorization
        X = self._add_service_type_features(X)
        
        # 8. Overall engagement score
        X = self._add_engagement_score(X)
        
        return X

    # --------------------------
    # Helper methods (modular)
    # --------------------------
    
    def _add_financial_features(self, X):
        """Add AvgChargePerMonth, CLV, and price hike flag."""
        # Avoid division by zero for tenure
        tenure_safe = X["tenure"].replace(0, 1)
        
        X["AvgChargePerMonth"] = X["TotalCharges"] / tenure_safe
        X["CLV"] = X["MonthlyCharges"] * X["tenure"]
        
        # Price hike detection
        avg_charge_safe = X["AvgChargePerMonth"].replace(0, np.nan)
        ratio = X["MonthlyCharges"] / avg_charge_safe
        X["Charge_Increase_Ratio"] = ratio.fillna(1.0)
        X["Recent_Price_Hike"] = (
            X["Charge_Increase_Ratio"] > self.price_hike_ratio
        ).astype(int)
        
        return X
    
    def _add_contract_payment_features(self, X):
        """Add contract type, auto-pay, and high-risk payment flags."""
        X["IsMonthContract"] = (X["Contract"] == "Month-to-month").astype(int)
        
        # Auto-pay: any payment method with "automatic" in name
        X["AutoPay"] = X["PaymentMethod"].str.contains("automatic", case=False).astype(int)
        
        # Interaction: paperless billing + auto-pay
        X["Paperless_AutoPay_Interaction"] = (
            (X["PaperlessBilling"] == "Yes").astype(int) * X["AutoPay"]
        )
        
        # High-risk combination: Electronic check + Month-to-month
        X["High_Risk_Payment_Contract"] = (
            (X["PaymentMethod"] == "Electronic check") & 
            (X["Contract"] == "Month-to-month")
        ).astype(int)
        
        return X
    
    def _add_service_usage_features(self, X):
        """Add add-on counts, multiple lines, and charges per service."""
        # Number of add-on services
        X["NumAddOnServices"] = (X[self._addon_cols] == "Yes").sum(axis=1)
        
        # Multiple lines flag
        X["HasMultipleLines"] = (X["MultipleLines"] == "Yes").astype(int)
        
        # Charges per service (avoid division by zero)
        num_services_safe = X["NumAddOnServices"].replace(0, 1)
        X["ChargesPerService"] = X["MonthlyCharges"] / num_services_safe
        
        return X
    
    def _add_family_demographic_features(self, X):
        """Add family presence and senior citizen risk flags."""
        # Family presence: partner or dependents
        X["HasFamily"] = (
            (X["Partner"] == "Yes") | (X["Dependents"] == "Yes")
        ).astype(int)
        
        # Senior citizen without family support (higher risk)
        X["FamilyRisk"] = (
            (X["SeniorCitizen"] == 1) & (X["HasFamily"] == 0)
        ).astype(int)
        
        return X
    
    def _add_tenure_features(self, X):
        """Add tenure groups and new/loyal customer flags."""
        # Tenure groups
        X["TenureGroup"] = pd.cut(
            X["tenure"],
            bins=self.tenure_bins,
            labels=self.tenure_labels,
            right=False
        ).astype(str)
        
        # New vs loyal customer flags
        X["IsNewCustomer"] = (X["tenure"] <= self.new_customer_threshold).astype(int)
        X["IsLoyalCustomer"] = (X["tenure"] >= self.loyal_customer_threshold).astype(int)
        
        return X
    
    def _add_protection_streaming_features(self, X):
        """Add protection and streaming bundle flags."""
        # Protection features
        X["HasProtection"] = (X[self._protection_cols] == "Yes").any(axis=1).astype(int)
        X["FullProtectionBundle"] = (
            X[self._protection_cols] == "Yes"
        ).all(axis=1).astype(int)
        
        # Streaming usage
        X["UsesStreaming"] = (X[self._streaming_cols] == "Yes").any(axis=1).astype(int)
        
        # Fiber optic internet flag
        X["FiberUser"] = (X["InternetService"] == "Fiber optic").astype(int)
        
        return X
    
    def _add_service_type_features(self, X):
        """Categorize service type: Both/PhoneOnly/InternetOnly/Neither."""
        has_phone = X["PhoneService"] == "Yes"
        has_internet = X["InternetService"] != "No"
        
        X["Service_Type"] = np.select(
            [
                has_phone & has_internet,
                has_phone & ~has_internet,
                ~has_phone & has_internet
            ],
            ["Both", "PhoneOnly", "InternetOnly"],
            default="Neither"
        )
        
        return X
    
    def _add_engagement_score(self, X):
        """Create a composite engagement score."""
        X["EngagementScore"] = (
            X["NumAddOnServices"] +
            X["HasProtection"] +
            X["UsesStreaming"] +
            X["AutoPay"]
        )
        return X

# Summary of Engineered Features

| Feature                       | Type        | Business Interpretation                              |
| ----------------------------- | ----------- | ---------------------------------------------------- |
| AvgChargePerMonth             | Numerical   | Average monthly customer spending                    |
| CLV                           | Numerical   | Estimated customer lifetime value                    |
| Charge_Increase_Ratio         | Numerical   | Current price relative to historical average         |
| Recent_Price_Hike             | Binary      | Indicates recent pricing increase                    |
| IsMonthContract               | Binary      | Customer has a month-to-month contract               |
| AutoPay                       | Binary      | Customer uses automatic payment                      |
| Paperless_AutoPay_Interaction | Numerical   | Digital payment engagement indicator                 |
| High_Risk_Payment_Contract    | Binary      | High-risk payment and contract combination           |
| NumAddOnServices              | Numerical   | Number of subscribed add-on services                 |
| HasMultipleLines              | Binary      | Customer has multiple phone lines                    |
| ChargesPerService             | Numerical   | Average monthly charge per subscribed add-on service |
| HasFamily                     | Binary      | Customer has partner and/or dependents               |
| FamilyRisk                    | Binary      | Senior customer without family support               |
| TenureGroup                   | Categorical | Customer lifecycle stage based on tenure             |
| IsNewCustomer                 | Binary      | Recently acquired customer                           |
| IsLoyalCustomer               | Binary      | Long-tenure loyal customer                           |
| HasProtection                 | Binary      | Uses at least one protection service                 |
| FullProtectionBundle          | Binary      | Uses all protection-related services                 |
| UsesStreaming                 | Binary      | Uses at least one streaming service                  |
| FiberUser                     | Binary      | Fiber optic internet subscriber                      |
| Service_Type                  | Categorical | Overall subscribed service portfolio                 |
| EngagementScore               | Numerical   | Composite indicator of customer engagement           |

In [4]:
# Load data
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Handle missing values in TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

# Separate features and target
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

In [5]:
BASE_NUMERIC_FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges']
BASE_BINARY_FEATURES = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
BASE_CATEGORICAL_FEATURES = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]

ENGINEERED_NUMERIC_FEATURES = [
    'AvgChargePerMonth', 'CLV', 'Charge_Increase_Ratio',
    'NumAddOnServices', 'ChargesPerService', 'EngagementScore'
]

ENGINEERED_BINARY_FEATURES = [
    'Recent_Price_Hike', 'IsMonthContract', 'AutoPay',
    'Paperless_AutoPay_Interaction', 'High_Risk_Payment_Contract',
    'HasMultipleLines', 'HasFamily', 'FamilyRisk',
    'IsNewCustomer', 'IsLoyalCustomer', 'HasProtection',
    'FullProtectionBundle', 'UsesStreaming', 'FiberUser'
]

ENGINEERED_CATEGORICAL_FEATURES = [
    'TenureGroup', 'Service_Type'
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('base_num', numeric_transformer, BASE_NUMERIC_FEATURES),
        ('eng_num', numeric_transformer, ENGINEERED_NUMERIC_FEATURES),
        ('base_binary', binary_transformer, BASE_BINARY_FEATURES),
        ('eng_binary', binary_transformer, ENGINEERED_BINARY_FEATURES),
        ('base_cat', categorical_transformer, BASE_CATEGORICAL_FEATURES),
        ('eng_cat', categorical_transformer, ENGINEERED_CATEGORICAL_FEATURES)
    ]
)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 5634
Test set size: 1409


In [7]:
fe = TelcoFeatureEngineer()

X_train_fe = fe.fit_transform(X_train)
X_test_fe = fe.transform(X_test)

print(X_train_fe.shape)
print(X_test_fe.shape)

(5634, 41)
(1409, 41)


In [8]:
numeric_features_base = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features_base = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]
preprocessor_base = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler())
        ]),
        numeric_features_base
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value",
                                       unknown_value=-1))
        ]),
        categorical_features_base
    )
])

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import pandas as pd

####################################################
# Baseline (بدون Feature Engineering)
####################################################

baseline_model = Pipeline([
    ("preprocessor", preprocessor_base),
    ("classifier", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

baseline_model.fit(X_train, y_train)

baseline_auc = roc_auc_score(
    y_test,
    baseline_model.predict_proba(X_test)[:,1]
)

####################################################
# Feature Engineering
####################################################

fe_model = Pipeline([
    ("feature_engineer", TelcoFeatureEngineer()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

fe_model.fit(X_train, y_train)

fe_auc = roc_auc_score(
    y_test,
    fe_model.predict_proba(X_test)[:,1]
)

####################################################
# تعداد ویژگی‌ها
####################################################

n_features_base = X_train.shape[1]

X_train_fe = TelcoFeatureEngineer().fit_transform(X_train)

n_features_fe = X_train_fe.shape[1]

####################################################
# جدول نهایی
####################################################

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Logistic Regression"
    ],
    "Feature Engineering": [
        "✗",
        "✓"
    ],
    "Train Shape": [
        f"{X_train.shape[0]}×{X_train.shape[1]}",
        f"{X_train_fe.shape[0]}×{X_train_fe.shape[1]}"
    ],
    "Test Shape": [
        f"{X_test.shape[0]}×{X_test.shape[1]}",
        f"{X_test.shape[0]}×{X_train_fe.shape[1]}"
    ],
    "Features": [
        n_features_base,
        n_features_fe
    ],
    "ROC-AUC": [
        round(baseline_auc,4),
        round(fe_auc,4)
    ]
})

display(comparison)

,Model,Feature Engineering,Train Shape,Test Shape,Features,ROC-AUC
0,Logistic Regression,✗,5634×19,1409×19,19,0.8397
1,Logistic Regression,✓,5634×41,1409×41,41,0.8460


In [21]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

# --- Cross-validation setup ---
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

def cv_evaluate_model(model, X, y, cv_obj):
    """
    Perform k-fold cross-validation and return mean ± std of metrics.
    """
    metrics_list = []
    
    for train_idx, val_idx in cv_obj.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Clone and fit model on fold train
        fold_model = clone(model)
        fold_model.fit(X_tr, y_tr)
        
        # Predict on fold validation
        y_pred = fold_model.predict(X_val)
        y_prob = fold_model.predict_proba(X_val)[:, 1]
        
        # Compute metrics
        metrics_list.append({
            "Accuracy": accuracy_score(y_val, y_pred),
            "Precision": precision_score(y_val, y_pred),
            "Recall": recall_score(y_val, y_pred),
            "F1-Score": f1_score(y_val, y_pred),
            "ROC-AUC": roc_auc_score(y_val, y_prob),
            "MCC": matthews_corrcoef(y_val, y_pred),
            "Log Loss": log_loss(y_val, y_prob)
        })
    
    # Aggregate: mean ± std
    df_metrics = pd.DataFrame(metrics_list)
    summary = {}
    for col in df_metrics.columns:
        mean_val = df_metrics[col].mean()
        std_val = df_metrics[col].std()
        summary[col] = f"{mean_val:.4f} ± {std_val:.4f}"
    
    return summary

# --- Evaluate baseline model with CV ---
baseline_cv_metrics = cv_evaluate_model(baseline_model, X_train, y_train, cv)

# --- Evaluate feature-engineered model with CV ---
fe_cv_metrics = cv_evaluate_model(fe_model, X_train, y_train, cv)

# --- Build comparison table ---
cv_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Feature Eng.": "✗",
        "Train Shape": f"{X_train.shape[0]}×{X_train.shape[1]}",
        "Features": X_train.shape[1],
        **baseline_cv_metrics
    },
    {
        "Model": "Logistic Regression",
        "Feature Eng.": "✓",
        "Train Shape": f"{X_train_fe.shape[0]}×{X_train_fe.shape[1]}",
        "Features": X_train_fe.shape[1],
        **fe_cv_metrics
    }
])

print("Cross-validation results (mean ± std over folds):")
display(cv_comparison)

# Optional: keep your original holdout test evaluation as well
# print("\nHoldout test results:")
# display(comparison)

Cross-validation results (mean ± std over folds):


,Model,Feature Eng.,Train Shape,Features,Accuracy,Precision,Recall,F1-Score,ROC-AUC,MCC,Log Loss
0,Logistic Regression,✗,5634×19,19,0.8023 ± 0.0127,0.6523 ± 0.0271,0.5458 ± 0.0432,0.5938 ± 0.0323,0.8444 ± 0.0130,0.4681 ± 0.0384,0.4186 ± 0.0130
1,Logistic Regression,✓,5634×41,41,0.8035 ± 0.0109,0.6662 ± 0.0263,0.5204 ± 0.0358,0.5839 ± 0.0282,0.8470 ± 0.0127,0.4642 ± 0.0330,0.4145 ± 0.0120


# Telco Churn Model – Evaluation Results

## Overview

This document summarizes the performance of Logistic Regression models trained on the Telco customer churn dataset, comparing:

- **Baseline model**: Uses only original features (19 total).
- **Feature‑engineered model**: Uses original features + engineered features (41 total).

Evaluation is performed using:

- **5‑fold stratified cross‑validation** (mean ± standard deviation),
- **Holdout test set** (20% of data).

---

## Cross‑Validation Results (5‑fold, mean ± std)

| Metric         | Baseline (19 features)     | With Feature Eng. (41 features) | Change      |
|----------------|-----------------------------|----------------------------------|-------------|
| **ROC‑AUC**    | 0.8444 ± 0.0130            | **0.8470 ± 0.0127**             | +0.0026     |
| **Accuracy**   | 0.8023 ± 0.0127            | **0.8035 ± 0.0109**             | +0.0012     |
| **Precision**  | 0.6523 ± 0.0271            | **0.6662 ± 0.0263**             | +0.0139     |
| **Recall**     | **0.5458 ± 0.0432**        | 0.5204 ± 0.0358                 | –0.0254     |
| **F1‑Score**   | **0.5938 ± 0.0323**        | 0.5839 ± 0.0282                 | –0.0099     |
| **MCC**        | **0.4681 ± 0.0384**        | 0.4642 ± 0.0330                 | –0.0039     |
| **Log Loss**   | 0.4186 ± 0.0130            | **0.4145 ± 0.0120**             | –0.0041     |


## Key Insights

### 1. ROC‑AUC and Log Loss improved

- **ROC‑AUC** increased slightly in both CV and holdout:
  - CV: 0.8444 → 0.8470
  - Holdout: 0.8397 → 0.8460
- **Log Loss** decreased:
  - CV: 0.4186 → 0.4145
  - Holdout: 0.4224 → 0.4144

**Interpretation:**  
The engineered features (`AvgChargePerMonth`, `CLV`, `EngagementScore`, etc.) add meaningful signal for ranking customers by churn risk and improve the calibration of predicted probabilities.

### 2. Precision improved, Recall decreased

- **Precision** (positive predictive value) improved:
  - CV: ~0.65 → ~0.66
  - Holdout: 0.6435 → 0.6644
- **Recall** (sensitivity) decreased:
  - CV: ~0.55 → ~0.52
  - Holdout: 0.5455 → 0.5187

**Interpretation:**  
With engineered features, the model becomes more conservative:
- When it predicts “churn”, it is more often correct (higher precision),
- But it misses more actual churners (lower recall).

### 3. F1‑Score slightly lower

- **F1‑Score** (harmonic mean of precision and recall) decreased slightly:
  - CV: 0.5938 → 0.5839
  - Holdout: 0.5904 → 0.5826

Because recall dropped more than precision improved, F1‑Score declined modestly.

### 4. Overall assessment

- **Feature engineering adds value**:
  - Better ranking (higher ROC‑AUC),
  - Better probability calibration (lower log loss),
  - Higher precision (fewer false positives).
- **Trade‑off**:
  - Lower recall means more missed churners.
  - If recall is critical, consider:
    - Adjusting the classification threshold,
    - Using more powerful models (e.g., Random Forest, XGBoost) that may better exploit these engineered features.

---

## Conclusion

The engineered features improve model performance in terms of **ranking** and **calibration**, but slightly reduce recall.  
This suggests that the feature engineering captures meaningful business signals (e.g., customer value, engagement, pricing dynamics), but further tuning or model selection may be needed if high recall is required.